In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv(r"C:\Users\drisy\Downloads\warehouse_order_processing.csv")

X = df.drop(columns=["Order_Processing_Time_min"])
y = df["Order_Processing_Time_min"]

feature_names = X.columns.tolist()

print("=" * 65)
print("         WAREHOUSE ORDER PROCESSING - REGRESSION ANALYSIS")
print("=" * 65)
print(f"Dataset shape: {df.shape}")

# ─────────────────────────────────────────────
# TRAIN-TEST SPLIT
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

# ─────────────────────────────────────────────
# STEP 1: SIMPLE LINEAR REGRESSION (SLR)
#         - One model per predictor
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 1: SIMPLE LINEAR REGRESSION (SLR) — Each Predictor vs Y")
print("=" * 65)

slr_results = []
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(feature_names):
    # Fit SLR using statsmodels for full stats
    X_slr = sm.add_constant(X_train[[col]])
    model_slr = sm.OLS(y_train, X_slr).fit()

    slope      = model_slr.params[col]
    intercept  = model_slr.params['const']
    r2         = model_slr.rsquared
    adj_r2     = model_slr.rsquared_adj
    p_val      = model_slr.pvalues[col]
    significant = "Yes ✓" if p_val < 0.05 else "No ✗"

    slr_results.append({
    "Predictor": col, "Intercept": round(intercept, 4),
    "Slope": round(slope, 4), "R²": round(r2, 4),
    "Adj R²": round(adj_r2, 4),           # ← add this line
    "P-value": round(p_val, 6), "Significant": significant
})
    # Scatter + regression line
    axes[i].scatter(X_train[col], y_train, alpha=0.3, s=10, color='steelblue')
    x_line = np.linspace(X_train[col].min(), X_train[col].max(), 100)
    axes[i].plot(x_line, intercept + slope * x_line, color='red', lw=2)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel("Processing Time", fontsize=8)
    axes[i].set_title(f"{col}\nR²={r2:.3f}, Adj R²={adj_r2:.3f}, p={p_val:.4f}", fontsize=9)


plt.suptitle("Simple Linear Regression — Each Predictor vs Processing Time",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("slr_plots.png", dpi=150, bbox_inches='tight')
plt.close()

slr_df = pd.DataFrame(slr_results)
print(slr_df.to_string(index=False))
print("\nSLR plots saved as 'slr_plots.png'")

# ─────────────────────────────────────────────
# STEP 2: MULTIPLE LINEAR REGRESSION (MLR)
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 2: MULTIPLE LINEAR REGRESSION (MLR) — Full Model")
print("=" * 65)

X_train_const = sm.add_constant(X_train)
X_test_const  = sm.add_constant(X_test)

full_model = sm.OLS(y_train, X_train_const).fit()
print(full_model.summary())

# Predictions
y_pred_train = full_model.predict(X_train_const)
y_pred_test  = full_model.predict(X_test_const)

train_r2   = r2_score(y_train, y_pred_train)
test_r2    = r2_score(y_test,  y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_test,  y_pred_test))
test_mae   = mean_absolute_error(y_test, y_pred_test)

print(f"\n{'Metric':<25} {'Train':>12} {'Test':>12}")
print("-" * 50)
print(f"{'R²':<25} {train_r2:>12.4f} {test_r2:>12.4f}")
print(f"{'RMSE':<25} {train_rmse:>12.4f} {test_rmse:>12.4f}")
print(f"{'MAE':<25} {'—':>12} {test_mae:>12.4f}")
with open("mlr_summary.txt", "w") as f:
    f.write(str(full_model.summary()))
print("MLR summary saved as 'mlr_summary.txt'")
# ─────────────────────────────────────────────
# STEP 3: OVERALL F-TEST
#         Tests if the full model is significant
#         H0: all coefficients = 0
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 3: OVERALL F-TEST (Full Model Significance)")
print("=" * 65)

f_stat   = full_model.fvalue
f_pvalue = full_model.f_pvalue
df_model = full_model.df_model
df_resid = full_model.df_resid

print(f"F-Statistic : {f_stat:.4f}")
print(f"Degrees of Freedom (Model) : {int(df_model)}")
print(f"Degrees of Freedom (Residual): {int(df_resid)}")
print(f"P-value     : {f_pvalue:.6f}")
print(f"\nConclusion  : {'Reject H0 — Model is statistically significant ✓' if f_pvalue < 0.05 else 'Fail to reject H0 — Model is NOT significant ✗'}")

# ─────────────────────────────────────────────
# STEP 4: PARTIAL F-TEST
#         Compare full model vs reduced model
#         (removing weak predictors)
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 4: PARTIAL F-TEST")
print("         Full Model vs Reduced Model")
print("         (Removing: Warehouse_Distance_m, Order_Weight_kg)")
print("=" * 65)

# Reduced model — drop the two weakest predictors
weak_vars    = ["Warehouse_Distance_m", "Order_Weight_kg"]
strong_vars  = [c for c in feature_names if c not in weak_vars]

X_train_reduced = sm.add_constant(X_train[strong_vars])
reduced_model   = sm.OLS(y_train, X_train_reduced).fit()

# Partial F-test formula:
# F = ((RSS_reduced - RSS_full) / q) / (RSS_full / df_resid_full)
RSS_full    = full_model.ssr
RSS_reduced = reduced_model.ssr
q           = len(weak_vars)          # number of removed predictors
df_resid_f  = full_model.df_resid

partial_F = ((RSS_reduced - RSS_full) / q) / (RSS_full / df_resid_f)
partial_p = 1 - stats.f.cdf(partial_F, dfn=q, dfd=df_resid_f)

print(f"\nFull Model    — RSS: {RSS_full:.4f},    R²: {full_model.rsquared:.4f}")
print(f"Reduced Model — RSS: {RSS_reduced:.4f}, R²: {reduced_model.rsquared:.4f}")
print(f"\nRemoved predictors (q={q}): {weak_vars}")
print(f"Partial F-Statistic : {partial_F:.4f}")
print(f"P-value             : {partial_p:.6f}")
print(f"\nConclusion: ", end="")
if partial_p < 0.05:
    print("Reject H0 — Removed variables ARE significant; keep full model ✓")
else:
    print("Fail to reject H0 — Removed variables are NOT significant; reduced model is sufficient ✓")

print("\n--- Reduced Model Summary ---")
print(reduced_model.summary())

# ─────────────────────────────────────────────
# STEP 5: RESIDUAL DIAGNOSTICS
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 5: RESIDUAL DIAGNOSTICS")
print("=" * 65)

residuals = full_model.resid
fitted    = full_model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.4, s=12, color='steelblue')
axes[0].axhline(0, color='red', lw=2, linestyle='--')
axes[0].set_xlabel("Fitted Values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted")

# Plot 2: Histogram of Residuals
axes[1].hist(residuals, bins=35, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_xlabel("Residuals")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of Residuals")

# Plot 3: Q-Q Plot
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist="norm")
axes[2].scatter(osm, osr, alpha=0.4, s=12, color='steelblue')
axes[2].plot(osm, slope * np.array(osm) + intercept, color='red', lw=2)
axes[2].set_xlabel("Theoretical Quantiles")
axes[2].set_ylabel("Sample Quantiles")
axes[2].set_title("Q-Q Plot of Residuals")

plt.suptitle("Residual Diagnostics — Full MLR Model",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("residual_diagnostics.png", dpi=150, bbox_inches='tight')
plt.close()
print("Residual diagnostic plots saved as 'residual_diagnostics.png'")

# Shapiro-Wilk test on residuals
_, sw_p = stats.shapiro(residuals.sample(200, random_state=42))
print(f"Shapiro-Wilk test on residuals: p = {sw_p:.4f} → "
      f"{'Residuals are normal ✓' if sw_p > 0.05 else 'Residuals deviate from normal ✗'}")

# ─────────────────────────────────────────────
# STEP 6: ACTUAL vs PREDICTED PLOT
# ─────────────────────────────────────────────
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_test, alpha=0.5, s=15, color='steelblue', label='Test Points')
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Fit')
plt.xlabel("Actual Processing Time (min)", fontsize=11)
plt.ylabel("Predicted Processing Time (min)", fontsize=11)
plt.title(f"Actual vs Predicted — Test Set (R²={test_r2:.4f})", fontsize=12, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=150, bbox_inches='tight')
plt.close()
print("Actual vs Predicted plot saved as 'actual_vs_predicted.png'")

# ─────────────────────────────────────────────
# STEP 7: COEFFICIENT COMPARISON PLOT
# ─────────────────────────────────────────────
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": full_model.params[1:].values,
    "P-value": full_model.pvalues[1:].values
}).sort_values("Coefficient")

colors = ['green' if p < 0.05 else 'salmon' for p in coef_df["P-value"]]

plt.figure(figsize=(9, 6))
bars = plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor='white')
plt.axvline(0, color='black', lw=1, linestyle='--')
plt.xlabel("Coefficient Value", fontsize=11)
plt.title("MLR Coefficients (Green = Significant, Red = Not Significant)",
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("coefficients_plot.png", dpi=150, bbox_inches='tight')
plt.close()
print("Coefficients plot saved as 'coefficients_plot.png'")

# ─────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(f"  Full MLR — Train R²  : {train_r2:.4f}")
print(f"  Full MLR — Test  R²  : {test_r2:.4f}")
print(f"  Full MLR — Test RMSE : {test_rmse:.4f} min")
print(f"  Full MLR — Test MAE  : {test_mae:.4f} min")
print(f"  Overall F-test p-val : {f_pvalue:.6f}  → Model significant ✓")
print(f"  Partial F-test p-val : {partial_p:.6f}  → {'Keep full model' if partial_p < 0.05 else 'Reduced model OK'}")
print("\nOutput files generated:")
print("  slr_plots.png | residual_diagnostics.png")
print("  actual_vs_predicted.png | coefficients_plot.png")

         WAREHOUSE ORDER PROCESSING - REGRESSION ANALYSIS
Dataset shape: (1000, 9)
Train size: 800 | Test size: 200

STEP 1: SIMPLE LINEAR REGRESSION (SLR) — Each Predictor vs Y
               Predictor  Intercept   Slope     R²  Adj R²  P-value Significant
       Num_Items_Ordered    51.8702  0.8203 0.3015  0.3006 0.000000       Yes ✓
    Warehouse_Distance_m    66.9712  0.0373 0.0214  0.0202 0.000032       Yes ✓
   Num_Workers_Available    85.4368 -1.2365 0.1025  0.1014 0.000000       Yes ✓
         Order_Weight_kg    64.6041  0.1645 0.0506  0.0494 0.000000       Yes ✓
             Hour_of_Day    69.6127  0.2675 0.0076  0.0064 0.013576       Yes ✓
      Num_Pending_Orders    54.1788  0.3754 0.2654  0.2644 0.000000       Yes ✓
Packing_Complexity_Score    59.9501  2.2915 0.1014  0.1003 0.000000       Yes ✓
  Equipment_Downtime_min    63.6920  0.6067 0.0649  0.0637 0.000000       Yes ✓

SLR plots saved as 'slr_plots.png'

STEP 2: MULTIPLE LINEAR REGRESSION (MLR) — Full Model
           